# 05. Memisahkan kode menjadi modul Python

Memahami pembagian tanggung jawab, membaca sumber fungsi, menjalankan pelatihan melalui terminal, dan memuat checkpoint.

**Prasyarat:** modul 01 sampai 04.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Dari notebook ke paket

Notebook cocok untuk menjelaskan dan menyelidiki. Paket Python menyediakan satu implementasi bersama sehingga perbaikan fungsi pelatihan tidak perlu disalin ke setiap notebook.

| Berkas | Tanggung jawab |
|---|---|
| data.py | Tokenisasi, kosakata, Dataset, batch |
| models.py | Definisi arsitektur |
| engine.py | Optimasi, evaluasi, checkpoint |
| train.py | Argumen terminal dan alur eksperimen |
| app.py | Masukan pengguna dan hasil prediksi |

In [2]:
print([p.name for p in (ROOT/"nlp_course").glob("*.py")])

['engine.py', 'data.py', 'train.py', '__init__.py', 'models.py']


## 2. Membaca fungsi yang digunakan

Abstraksi seharusnya dapat dibuka. `inspect.getsource` memperlihatkan implementasi yang benar-benar digunakan kernel. Cocokkan setiap tahap dengan siklus manual pada modul 01.

In [3]:
import inspect
from nlp_course.engine import run_epoch
print(inspect.getsource(run_epoch))

def run_epoch(model, loader, optimizer=None, device='cpu'):
    training = optimizer is not None
    model.train(training)
    total_loss, n, truth, preds = 0., 0, [], []
    criterion = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(training):
        for ids, lengths, labels in loader:
            ids, labels = ids.to(device), labels.to(device)
            logits = model(ids, lengths)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.)
                optimizer.step()
            total_loss += loss.item() * len(labels)
            n += len(labels)
            truth.extend(labels.cpu().tolist())
            preds.extend(logits.argmax(1).detach().cpu().tolist())
    if n == 0:
        raise ValueError('DataLoader kosong')
    return {'loss': total_loss / n, **metrics(truth, preds)}



## 3. Memeriksa antarmuka model

Setiap model menerima ids dan lengths lalu menghasilkan logits `[B,C]`. Keseragaman antarmuka memungkinkan model diganti tanpa menulis ulang siklus pelatihan.

In [4]:
vocab,train,val,test = loaders()
ids,lengths,labels = next(iter(train))
for model in [MeanClassifier(len(vocab)),RecurrentClassifier(len(vocab)),TinyTransformer(len(vocab))]:
    output = model(ids,lengths)
    print(type(model).__name__,output.shape)
    assert output.shape == (len(labels),2)

MeanClassifier torch.Size([16, 2])
RecurrentClassifier torch.Size([16, 2])
TinyTransformer torch.Size([16, 2])


## 4. Pelatihan melalui terminal

Pemanggilan subprocess berikut setara dengan menjalankan `python -m nlp_course.train`. Setiap proses memulai lingkungan Python baru. Perintah mengeluarkan riwayat validasi dan metrik uji setelah pemilihan checkpoint.

In [5]:
import subprocess
result = subprocess.run([sys.executable,"-m","nlp_course.train","--epochs","12",
                         "--output",str(ARTIFACTS/"sentiment.pt")],
                        cwd=ROOT,capture_output=True,text=True,check=True)
print(result.stdout[-1000:])

 0.7333333492279053
    },
    {
      "epoch": 8,
      "train_loss": 0.37905532121658325,
      "val_loss": 0.4353477309147517,
      "val_macro_f1": 0.7333333492279053
    },
    {
      "epoch": 9,
      "train_loss": 0.3744238176279598,
      "val_loss": 0.43350958327452344,
      "val_macro_f1": 0.7333333492279053
    },
    {
      "epoch": 10,
      "train_loss": 0.3680724923809369,
      "val_loss": 0.4392220725615819,
      "val_macro_f1": 0.7333333492279053
    },
    {
      "epoch": 11,
      "train_loss": 0.36965106841590667,
      "val_loss": 0.4398677895466487,
      "val_macro_f1": 0.7333333492279053
    },
    {
      "epoch": 12,
      "train_loss": 0.36867912279234993,
      "val_loss": 0.42672891914844513,
      "val_macro_f1": 0.7333333492279053
    }
  ],
  "test": {
    "loss": 0.4348430633544922,
    "accuracy": 0.75,
    "macro_f1": 0.7333333492279053,
    "confusion": [
      [
        24,
        24
      ],
      [
        0,
        48
      ]
    ]
  }
}


## 5. Memuat model tanpa melatih ulang

Pipeline prediksi harus menggunakan tokenizer, kosakata, panjang maksimum, dan urutan label yang disimpan saat pelatihan. Mengubah salah satunya dapat menghasilkan prediksi yang keliru walaupun bobot berhasil dimuat.

In [6]:
model, metadata = load_mean(ARTIFACTS/"sentiment.pt")
print(metadata.keys())
print(predict("buku ini baik",model,metadata))

dict_keys(['state_dict', 'vocab', 'dim', 'classes', 'max_length', 'tokenizer', 'labels'])
{'negatif': 0.43367090821266174, 'positif': 0.5663290619850159}


## 6. Konfigurasi dan reproduksibilitas

Catat seed, versi PyTorch, jumlah epoch, pembagian dataset, dan sumber data. Checkpoint menyimpan informasi prapemrosesan untuk inferensi. Riwayat eksperimen perlu disimpan terpisah agar keputusan pelatihan dapat diaudit.

In [7]:
import json
config = {"seed":42,"epochs":12,"torch":str(torch.__version__),
          "dataset":"data/reviews.csv","model":"MeanClassifier","embedding_dim":32}
(ARTIFACTS/"config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")
print(config)

{'seed': 42, 'epochs': 12, 'torch': '2.14.0+cu130', 'dataset': 'data/reviews.csv', 'model': 'MeanClassifier', 'embedding_dim': 32}


## Latihan mandiri

1. Di mana Anda mengganti tokenizer?
2. Di mana Anda menambahkan arsitektur baru?
3. Mengapa bobot dan kosakata harus dipasangkan?
4. Jalankan CLI dengan seed 7 lalu bandingkan validasi.

## Pembahasan latihan

1. data.py. Versi tokenizer di metadata checkpoint juga harus berubah bila aturan berubah.
2. models.py dengan antarmuka forward yang sama.
3. Satu indeks harus menunjuk token yang sama saat pelatihan dan prediksi.
4. Perbedaan seed dapat menghasilkan parameter dan metrik yang berbeda. Gunakan beberapa seed untuk membandingkan model.

## Penghubung ke materi berikutnya

Modul 06 memakai kembali bobot dari domain sumber untuk mempelajari domain sasaran.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.